# 02 · Preprocessing — Flight Price (Regression)
Fixed lookups run once before CV; steps that learn from data run inside the CV pipeline. Code: `src/regression/preprocess.py`.

## Setup
Make `src/` importable and load the shared config (seed, paths).

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

# Make `src` importable: repo root locally, or an uploaded Kaggle dataset containing src/
for p in [Path.cwd(), *Path.cwd().parents, *(f.parent.parent for f in Path("/kaggle/input").glob("**/src/config.py"))]:
    if (p / "src" / "config.py").exists():
        sys.path.insert(0, str(p.resolve()))
        break

from src import config

Load the raw data.

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

from src import evaluate
from src.regression import preprocess as pp
from src.regression.models import GroupMedianRegressor

TASK = "regression"
pd.set_option("display.width", 140)

train_raw, test_raw, sample_sub = config.load(TASK)
print("raw train", train_raw.shape, "| raw test", test_raw.shape)

raw train (15589, 12) | raw test (3898, 11)


## Step 1 — Recover `airline` from the flight-code prefix

In [3]:
prefix_map = pp.build_prefix_map(train_raw, test_raw)
print(len(prefix_map), "prefixes")
prefix_map.sort_index().to_frame("airline").T

20 prefixes


prefix,6E,9I,AI,BG,EK,EY,FZ,GF,IX,KU,MH,QP,QR,RA,S5,SG,SQ,TG,UL,WY
airline,Oceanic Airlines,Massive Dynamic,Wayfarer Aviation,Tyrell Aeronautics,Stark Aviation,Nakatomi Airlines,Initech Air,Aperture Aviation,Wayfarer Aviation,Black Mesa Air,Virtucon,Sandpiper Air,Vault-Tec Air,Cyberdyne Flights,Hooli Airways,Vandelay Aviation,Weyland Air,Wayne Enterprises Air,Ajira Airways,Omni Consumer Flights


Missing airlines before and after.

In [4]:
train_air = pp.recover_airline(train_raw, prefix_map)
test_air = pp.recover_airline(test_raw, prefix_map)

pd.DataFrame({"before": [train_raw.airline.isna().sum(), test_raw.airline.isna().sum()],
              "after": [train_air.airline.isna().sum(), test_air.airline.isna().sum()]},
             index=["train", "test"])

,before,after
train,3089,0
test,799,0


- All missing airlines filled.

## Step 2 — Drop exact duplicates (after Step 1)

In [5]:
cols = [c for c in train_raw.columns if c != pp.ID]
dups_before = train_raw.duplicated(subset=cols).sum()
dups_after_recovery = train_air.duplicated(subset=cols).sum()
print(f"exact duplicates in raw data:              {dups_before}")
print(f"exact duplicates after airline recovery:   {dups_after_recovery}")

train_clean = pp.drop_exact_duplicates(train_air)
print(f"train rows: {len(train_raw)} -> {len(train_clean)}")

exact duplicates in raw data:              279
exact duplicates after airline recovery:   422
train rows: 15589 -> 15167


- 279 duplicates in the raw data, 422 after airline recovery; 15,589 → 15,167 rows.

`load_clean()` applies Steps 1–2 in one call.

In [6]:
X, y, X_test, test_ids, sample_sub = pp.load_clean()
assert len(X) == len(train_clean) and X.airline.isna().sum() == 0
print("X", X.shape, "| y", y.shape, "| X_test", X_test.shape)
print("feature columns:", list(X.columns))

X (15167, 10) | y (15167,) | X_test (3898, 10)
feature columns: ['airline', 'flight', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'flight_class', 'duration', 'days_left']


## Step 3 — Fill missing `duration` (fitted inside CV)
How many values each rule fills.

In [7]:
imp = pp.DurationImputer().fit(X)
pd.DataFrame({"train": imp.fill_report(X), "test": imp.fill_report(X_test)})

,train,test
missing,3065,776
group median,2961,752
route+stops median,104,24
global median,0,0


Check: hide 15% of known durations and fill them back.

In [8]:
rng = np.random.default_rng(config.SEED)
known_idx = X.index[X.duration.notna()]
hide = rng.choice(known_idx, size=int(0.15 * len(known_idx)), replace=False)

X_masked = X.copy()
X_masked.loc[hide, "duration"] = np.nan
filled = pp.DurationImputer().fit(X_masked).transform(X_masked).loc[hide, "duration"]
truth = X.loc[hide, "duration"]

err = (filled - truth).abs()
print(f"hidden values: {len(hide)}")
print(f"exactly right (within 0.01 h): {(err < 0.01).mean():.1%}")
print(f"mean absolute error: {err.mean():.2f} h | median: {err.median():.2f} h | 95th pct: {err.quantile(0.95):.2f} h")
print(f"for comparison, filling with one overall median would give MAE {(truth - X_masked.duration.median()).abs().mean():.2f} h")

hidden values: 1815
exactly right (within 0.01 h): 86.3%
mean absolute error: 0.34 h | median: 0.00 h | 95th pct: 1.00 h
for comparison, filling with one overall median would give MAE 2.25 h


- 86% filled exactly, mean error 0.34 h (2.25 h with one overall median).

## Step 4 — Encode features

In [9]:
pre = pp.make_preprocessor()
display(pre)

M = pre.fit_transform(X)
M_test = pre.transform(X_test)
print("train matrix", M.shape, "| test matrix", M_test.shape)
print("missing values left — train:", int(M.isna().sum().sum()), "| test:", int(M_test.isna().sum().sum()))
assert list(M.columns) == list(M_test.columns)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('duration', ...), ('columns', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('stops', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature

train matrix (15167, 40) | test matrix (3898, 40)
missing values left — train: 0 | test: 0


Encoded column groups and the 'infrequent' airline bucket.

In [10]:
groups = pd.Series(M.columns).str.extract(r"^(airline|source_city|destination_city|departure_time|arrival_time|flight_class)")[0]
print(groups.fillna("numeric / ordinal").value_counts().to_string())

enc = pre.named_steps["columns"].named_transformers_["airline"]
print("\nairlines in the 'infrequent' column:", list(enc.infrequent_categories_[0]))
new_in_test = sorted(set(X_test.airline) - set(X.airline))
print("test-only airlines:", new_in_test, "-> infrequent column =",
      M_test.loc[X_test.airline.isin(new_in_test).to_numpy(), "airline_infrequent_sklearn"].tolist())

0
airline              11
source_city           6
destination_city      6
arrival_time          6
departure_time        6
numeric / ordinal     3
flight_class          2

airlines in the 'infrequent' column: ['Aperture Aviation', 'Black Mesa Air', 'Cyberdyne Flights', 'Hooli Airways', 'Initech Air', 'Massive Dynamic', 'Omni Consumer Flights', 'Weyland Air']
test-only airlines: ['Tyrell Aeronautics'] -> infrequent column = [1.0]


- 40 numeric columns; airlines with < 50 rows share one 'infrequent' column.

Scaled columns: train vs test statistics.

In [11]:
num_cols = ["duration", "days_left", "stops"]
pd.DataFrame({"train mean": M[num_cols].mean(), "train std": M[num_cols].std(),
              "test mean": M_test[num_cols].mean(), "test std": M_test[num_cols].std()}).round(3)

,train mean,train std,test mean,test std
duration,-0.0,1.0,-0.010,0.982
days_left,-0.0,1.0,-0.005,1.014
stops,-0.0,1.0,-0.007,0.994


## Step 5 — Baselines (5-fold CV, seed 42)

In [12]:
baselines = {
    "baseline_global_median": GroupMedianRegressor(),
    "baseline_class_median": GroupMedianRegressor(group_col="flight_class"),
}
results = {}
for name, model in baselines.items():
    folds = evaluate.cross_validate(model, X, y, TASK, name, params=model.get_params(),
                                    notes="Phase 2 baseline")
    results[name] = evaluate.summarize(folds)
pd.DataFrame(results).T

,rmse,mae,r2,rmsle,train_r2
baseline_global_median,43962.7057 ± 1037.3956,22321.1036 ± 783.7697,-0.2502 ± 0.0162,1.1177 ± 0.0225,-0.2500 ± 0.0022
baseline_class_median,33151.5727 ± 854.8586,13397.1486 ± 475.4353,0.2891 ± 0.0097,0.4978 ± 0.0083,0.2891 ± 0.0029


- Class-median baseline: CV R² 0.289.

Experiment log.

In [13]:
pd.read_csv(config.results_dir(TASK) / "experiments.csv")

,timestamp,model,params,n_rows,rmse_mean,rmse_std,mae_mean,mae_std,r2_mean,r2_std,rmsle_mean,rmsle_std,notes,public_lb,train_r2_mean,train_r2_std
0,2026-10-11T20:01:58,1_ols,{},15167,15996.632983,412.782847,7783.798392,185.181252,0.834460,0.003817,1.047947,0.053598,plain least squares,NaN,0.836606,0.002726
1,2026-10-11T20:01:59,2_ols_log,{},15167,16372.384859,613.942224,6414.835920,208.247822,0.826403,0.012001,0.262359,0.004240,log target,NaN,0.827468,0.001668
2,2026-10-11T20:02:01,3_ols_class_interactions,{},15167,13592.696252,538.308103,5738.473876,229.748259,0.880500,0.005105,0.263668,0.023120,every feature x class indicator,NaN,0.882622,0.002282
3,2026-10-11T20:02:02,4_ols_per_class,{},15167,13827.868051,513.871368,5838.174398,224.958217,0.876313,0.005185,0.264791,0.021689,one OLS per flight_class,NaN,0.878486,0.003557
4,2026-10-11T20:02:05,5_poly2_ols,{},15167,11977.399930,493.308283,5443.705890,82.845100,0.906978,0.008396,0.860953,0.070478,degree-2 polynomial features,NaN,0.932885,0.001538
5,2026-10-11T20:02:14,6_poly2_ridge,"{""alpha"": 10}",15167,11291.575501,370.661639,5282.247159,87.064365,0.917383,0.006049,0.799594,0.159525,degree-2 polynomial + L2 | best of 4 settings,0.94450,0.929122,0.001495
6,2026-10-11T20:02:57,7_lasso_class_interactions,"{""alpha"": 10}",15167,13600.900079,545.844160,5736.375749,233.898712,0.880354,0.005270,0.266057,0.022070,class interactions + L1 | best of 3 settings,NaN,0.882379,0.002278
7,2026-10-11T20:03:07,8_tree_full,{},15167,14532.793867,597.918031,4052.977274,177.735083,0.863350,0.007383,0.178844,0.004107,unpruned,NaN,0.999312,0.000093
8,2026-10-11T20:03:29,9_tree_pruned,"{""max_depth"": 12, ""min_samples_leaf"": 5, ""ccp_...",15167,11957.491197,668.971259,4091.126339,179.937272,0.907275,0.009694,0.164392,0.002530,max_depth/min_samples_leaf/ccp_alpha | best of...,NaN,0.938986,0.002641
9,2026-10-11T20:28:24,6b_poly2_ridge_floor,"{""alpha"": 10}",15167,11234.052207,382.268198,5176.807966,101.279729,0.918228,0.006002,0.308011,0.015624,Phase 3 best + floor at min train price,NaN,0.929710,0.001511


## Step 6 — Baseline submission

In [14]:
path, sub = evaluate.make_submission(baselines["baseline_class_median"], X, y, X_test, test_ids,
                                     sample_sub, TASK, "baseline_class_median", date="2026-10-11")
print("saved:", path.relative_to(config.ROOT) if not config.ON_KAGGLE else path)
print("shape:", sub.shape, "| columns:", list(sub.columns))
print(sub[sub.columns[1]].value_counts().rename("rows").to_string())
sub.head()

saved: submissions\regression\2026-10-11_baseline_class_median.csv
shape: (3898, 2) | columns: ['id', 'price']
price
7580.0     2337
38546.0    1561


,id,price
0,1,7580.0
1,2,7580.0
2,3,38546.0
3,4,38546.0
4,5,7580.0


Expected Kaggle score for each candidate metric.

In [15]:
guide = evaluate.cross_validate(baselines["baseline_class_median"], X, y, TASK, "", log=False).mean()
pd.DataFrame({"expected Kaggle score if metric is…": guide.round(4)})

,expected Kaggle score if metric is…
rmse,33151.5727
mae,13397.1486
r2,0.2891
rmsle,0.4978
train_r2,0.2891


- Kaggle public score 0.31495 matches CV R², so the metric is **R² on raw price**.